    # Simple ESG Carbon RAG (Colab-Friendly Teaching Notebook)

    In this notebook, We learn how RAG works in practice with a beginner-friendly workflow.

    We build a **simple RAG pipeline** to extract carbon-emissions-related information from ESG / sustainability reports for:
    - Amazon
    - Google
    - Apple
    - Meta

    ## Colab Quick Start
    1. We open this notebook in Google Colab.
    2. We run the Colab setup cells first.
    3. We use Colab's built-in AI model (`google.colab.ai`) for generation, so We do not need an external API key.
    4. We run one company first, then the four-company batch.

    ## Teaching goal
    We use this flow:
    1. Parse report text
    2. Split into chunks
    3. Store chunks in a vector database
    4. Retrieve relevant chunks for a question
    5. Ask Colab's built-in LLM to extract structured answers from retrieved evidence
    


## Practical Context: Green Finance / ESG Research

Analysts often need to extract:
- Scope 1 / Scope 2 / Scope 3 emissions
- Total GHG emissions
- Emissions intensity
- Carbon reduction targets
- Reporting year and boundary notes

RAG helps because ESG reports are long and noisy.
Instead of sending the whole report to an LLM, we retrieve only the most relevant passages.


    ## Dependencies (Colab-Friendly Setup)

    For Google Colab, We run the setup cells below instead of manually installing everything.

    Important:
    - We use Colab's built-in AI model (`google.colab.ai`) for text generation (no external API key).
    - We install local packages for parsing PDFs, building embeddings, and running ChromaDB retrieval.
    - We usually keep Colab's preinstalled `torch` unchanged.
    - This notebook is tested locally with Python `3.12.10`, and this Colab version is designed to run in Colab's managed Python environment.
    


    ## Colab Runtime Setup (Colab AI + Packages)

    We run this section first in Colab.

    - The LLM call uses Colab's built-in AI service (`google.colab.ai`), so We do not configure an API key.
    - We keep Colab's preinstalled `torch`.
    - We install the remaining packages (`chromadb`, `sentence-transformers`, etc.) for the local RAG pipeline.
    - GPU is optional here. Colab AI generation does not depend on our local GPU, but GPU can still help embeddings run faster.
    


In [ ]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
RUN_COLAB_INSTALL = True

print("IN_COLAB =", IN_COLAB)
print("Python version =", sys.version.split()[0])

if IN_COLAB and RUN_COLAB_INSTALL:
    packages = [
        "pypdf==6.7.3",
        "pandas==3.0.1",
        "chromadb==1.5.1",
        "sentence-transformers==5.2.3",
        "python-dotenv==1.2.1",
        "requests==2.32.5",
    ]
    print("Installing Colab packages (keeping preinstalled torch)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
    print("Colab package installation complete.")
else:
    print("Skipping Colab package installation.")

if IN_COLAB:
    print("\nColab AI note: This notebook uses google.colab.ai (no external API key).")



In [ ]:
COLAB_AI_AVAILABLE = False
COLAB_AI_IMPORT_ERROR = None

try:
    import torch
    print("torch version =", torch.__version__)
    print("CUDA available =", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU =", torch.cuda.get_device_name(0))
    else:
        print("GPU is optional for this notebook. Colab AI generation still works without a local GPU.")
except Exception as e:
    print("Torch check skipped:", e)

if IN_COLAB:
    try:
        from google.colab import ai as _colab_ai_probe  # noqa: F401
        COLAB_AI_AVAILABLE = True
        print("google.colab.ai is available in this runtime.")
    except Exception as e:
        COLAB_AI_IMPORT_ERROR = str(e)
        print("google.colab.ai is not available:", e)
        print("Open this notebook in Google Colab and use a standard Colab runtime.")
else:
    print("Not running in Colab. google.colab.ai is not available in local notebooks.")



## Data Folder (Expected)

```text
RAG/
  simple_rag_esg_carbon_teaching.ipynb
  data/
    esg_reports/
      amazon_2024_sustainability_report.pdf
      apple_2025_environmental_progress_report.pdf
      google_2025_environmental_report.pdf
      meta_2025_sustainability_report.pdf
```

Note:
- The parser in this notebook supports both `.pdf` and `.md/.txt`.


## Optional: Persist Files in Google Drive (Colab)

Colab local storage (`/content`) is temporary.
If we want to keep downloaded reports, vector DB files, and outputs after the session ends, we can mount Google Drive and set `PROJECT_ROOT`.


In [ ]:
USE_GOOGLE_DRIVE = False
GOOGLE_DRIVE_PROJECT_DIR = "/content/drive/MyDrive/ESG_RAG_colab"
import sys

if "google.colab" in sys.modules and USE_GOOGLE_DRIVE:
    from google.colab import drive
    import os
    drive.mount("/content/drive")
    os.makedirs(GOOGLE_DRIVE_PROJECT_DIR, exist_ok=True)
    os.environ["PROJECT_ROOT"] = GOOGLE_DRIVE_PROJECT_DIR
    print("PROJECT_ROOT set to:", os.environ["PROJECT_ROOT"])
else:
    print("Using current working directory. PROJECT_ROOT is not overridden.")


In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys
import re
import json
import time
from datetime import datetime
from typing import Any, Dict, List

import pandas as pd
import requests
from pypdf import PdfReader
from dotenv import load_dotenv

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

load_dotenv()

IN_COLAB = "google.colab" in sys.modules
PROJECT_ROOT = Path(os.getenv("PROJECT_ROOT", ".")).resolve()
DATA_DIR = PROJECT_ROOT / "data" / "esg_reports"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
VECTOR_DB_DIR = PROJECT_ROOT / "vector_db" / "chroma_esg"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
VECTOR_DB_DIR.mkdir(exist_ok=True, parents=True)

print("IN_COLAB:", IN_COLAB)
print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())
print("Vector DB dir:", VECTOR_DB_DIR.resolve())



## Optional Download Helper (for reproducibility)

We use these URLs in this run.
In Colab, we keep the remote download option and default to downloading missing files automatically.
We can still set `DOWNLOAD_MISSING = False` if the files are already available.

Important:
- We use official report links for the four target companies in this notebook run.


In [ ]:
DOWNLOAD_MISSING = bool(globals().get("IN_COLAB", False))
# Colab default: download missing reports automatically.
# Local default: keep False unless we want to download.
# We can override manually, for example:
# DOWNLOAD_MISSING = False

SOURCE_URLS = {
    "amazon_2024_sustainability_report.pdf": "https://sustainability.aboutamazon.com/2024-amazon-sustainability-report.pdf",
    "apple_2025_environmental_progress_report.pdf": "https://www.apple.com/environment/pdf/Apple_Environmental_Progress_Report_2025.pdf",
    "google_2025_environmental_report.pdf": "https://www.gstatic.com/gumdrop/sustainability/google-2025-environmental-report.pdf",
    "meta_2025_sustainability_report.pdf": "https://sustainability.atmeta.com/wp-content/uploads/2025/08/Meta_2025-Sustainability-Report_.pdf",
}

if DOWNLOAD_MISSING:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    headers = {"User-Agent": "Mozilla/5.0"}
    for name, url in SOURCE_URLS.items():
        path = DATA_DIR / name
        if path.exists() and path.stat().st_size > 100_000:
            print("SKIP", name)
            continue
        print("Downloading", name)
        r = requests.get(url, headers=headers, timeout=300)
        r.raise_for_status()
        if name.endswith(".pdf"):
            path.write_bytes(r.content)
        else:
            path.write_text(r.text, encoding="utf-8")
        print("Saved", name, path.stat().st_size, "bytes")
else:
    print("DOWNLOAD_MISSING = False (using local files if present)")

with open(OUTPUT_DIR / "download_sources.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "generated_at": datetime.utcnow().isoformat() + "Z",
            "source_urls": SOURCE_URLS,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )
print("Saved source URL record to", OUTPUT_DIR / "download_sources.json")


## Step 1. Load Documents (PDF + Text Mirror Support)

We keep parsing lightweight:
- `pypdf` for normal PDFs
- direct text loading for `.md/.txt`

For text mirror files, we remove the `r.jina.ai` header and keep the report content section.


In [ ]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    reader = PdfReader(str(pdf_path))
    parts = []
    for page in reader.pages:
        parts.append(page.extract_text() or "")
    return "\n".join(parts)


def extract_text_from_text_file(path: Path) -> str:
    text = path.read_text(encoding="utf-8", errors="ignore")
    # r.jina.ai PDF mirror format includes a header and then "Markdown Content:"
    marker = "Markdown Content:"
    if marker in text:
        text = text.split(marker, 1)[1].strip()
    return text


def infer_company_label(file_name: str) -> str:
    lower = file_name.lower()
    if lower.startswith("amazon_"):
        return "Amazon"
    if lower.startswith("google_"):
        return "Google"
    if lower.startswith("apple_"):
        return "Apple"
    if lower.startswith("meta_"):
        return "Meta"
    return Path(file_name).stem


def load_documents(data_dir: Path) -> List[Dict[str, Any]]:
    docs: List[Dict[str, Any]] = []
    for path in sorted(data_dir.iterdir()):
        if path.name.lower().startswith("tesla_"):
            continue
        if path.suffix.lower() not in {".pdf", ".md", ".txt"}:
            continue
        try:
            if path.suffix.lower() == ".pdf":
                text = extract_text_from_pdf(path)
                source_type = "pdf"
            else:
                text = extract_text_from_text_file(path)
                source_type = "text_mirror"
        except Exception as e:
            print(f"SKIP (parse error): {path.name} -> {e}")
            continue

        docs.append(
            {
                "doc_id": path.stem,
                "company": infer_company_label(path.name),
                "file_name": path.name,
                "source_type": source_type,
                "text": text,
                "char_count": len(text),
            }
        )
    return docs


documents = load_documents(DATA_DIR) if DATA_DIR.exists() else []
print(f"Loaded {len(documents)} document(s).")
for d in documents:
    print(f"- {d['company']}: {d['file_name']} ({d['source_type']}) | {d['char_count']:,} chars")


## Step 2. Chunk the Documents

We split report text into chunks before storing them in the vector database.

For faster embedding in Colab, We use a speed-focused chunk setup:
- larger chunks (fewer chunks total)
- smaller overlap
- same simple paragraph-aware chunker


In [ ]:
def normalize_text(text: str) -> str:
    text = text.replace("\u00a0", " ")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# Speed-focused chunking defaults for Colab:
# - larger chunks -> fewer embeddings to compute
# - smaller overlap -> less duplicated text across chunks
CHUNK_MAX_CHARS = 1400
CHUNK_OVERLAP_CHARS = 60


def chunk_text(
    text: str,
    max_chars: int = CHUNK_MAX_CHARS,
    overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> List[str]:
    text = normalize_text(text)
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks: List[str] = []
    current = ""

    for para in paragraphs:
        candidate = f"{current}\n\n{para}".strip() if current else para
        if len(candidate) <= max_chars:
            current = candidate
            continue

        if current:
            chunks.append(current)
            current = ""

        if len(para) <= max_chars:
            current = para
        else:
            start = 0
            while start < len(para):
                end = start + max_chars
                piece = para[start:end]
                chunks.append(piece)
                if end >= len(para):
                    break
                start = max(0, end - overlap_chars)

    if current:
        chunks.append(current)

    # lightweight overlap between adjacent chunks
    if overlap_chars > 0 and len(chunks) > 1:
        out = [chunks[0]]
        for i in range(1, len(chunks)):
            prefix = chunks[i - 1][-overlap_chars:]
            out.append((prefix + "\n" + chunks[i]).strip())
        chunks = out

    return chunks


def build_chunk_records(
    docs: List[Dict[str, Any]],
    max_chars: int = CHUNK_MAX_CHARS,
    overlap_chars: int = CHUNK_OVERLAP_CHARS,
) -> List[Dict[str, Any]]:
    records: List[Dict[str, Any]] = []
    for doc in docs:
        chunks = chunk_text(doc["text"], max_chars=max_chars, overlap_chars=overlap_chars)
        for i, chunk in enumerate(chunks):
            records.append(
                {
                    "id": f"{doc['doc_id']}::chunk::{i}",
                    "doc_id": doc["doc_id"],
                    "company": doc["company"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                    "chunk_id": i,
                    "text": chunk,
                }
            )
    return records


chunk_records = build_chunk_records(
    documents,
    max_chars=CHUNK_MAX_CHARS,
    overlap_chars=CHUNK_OVERLAP_CHARS,
)
print("Chunk config:", {"max_chars": CHUNK_MAX_CHARS, "overlap_chars": CHUNK_OVERLAP_CHARS})
print("Total chunks:", len(chunk_records))
if chunk_records:
    print("Example chunk:", chunk_records[0]["id"])
    print(chunk_records[0]["text"][:600])


## Step 3. Build a Vector Database (ChromaDB)

We store chunks in a vector database using:
- `ChromaDB` (local persistent vector DB)
- a smaller embedding model for speed (`paraphrase-MiniLM-L3-v2`)

Speed changes in this version of the Colab notebook:
- default `REBUILD_VECTOR_DB = False` (reuse existing index)
- try GPU for embedding in Colab when CUDA is available
- skip re-indexing when the collection already exists and has data


In [ ]:
EMBED_MODEL_NAME = "sentence-transformers/paraphrase-MiniLM-L3-v2"
COLLECTION_NAME = "esg_carbon_chunks_fast_l3_colab_v1"
REBUILD_VECTOR_DB = False  # build once, then reuse unless We intentionally rebuild


def detect_embedding_device() -> str:
    try:
        import torch
        if IN_COLAB and torch.cuda.is_available():
            return "cuda"
    except Exception:
        pass
    return "cpu"


EMBEDDING_DEVICE = detect_embedding_device()
print("Embedding model:", EMBED_MODEL_NAME)
print("Embedding device:", EMBEDDING_DEVICE)
print("Collection name:", COLLECTION_NAME)
print("REBUILD_VECTOR_DB:", REBUILD_VECTOR_DB)


def make_embedding_function(model_name: str, device: str):
    # Chroma's SentenceTransformerEmbeddingFunction supports `device` in recent versions.
    # We keep a fallback for compatibility with older versions.
    try:
        print(f"Creating embedding function on device={device} ...")
        return SentenceTransformerEmbeddingFunction(model_name=model_name, device=device)
    except TypeError:
        print("This Chroma version does not accept device=... in SentenceTransformerEmbeddingFunction.")
        print("Falling back to default constructor (embedding may run on CPU).")
        return SentenceTransformerEmbeddingFunction(model_name=model_name)


embedding_fn = make_embedding_function(EMBED_MODEL_NAME, EMBEDDING_DEVICE)
chroma_client = chromadb.PersistentClient(path=str(VECTOR_DB_DIR))

if REBUILD_VECTOR_DB:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
        print("Deleted existing collection:", COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"},
)


def index_chunks_to_chroma(collection, chunk_records: List[Dict[str, Any]], batch_size: int = 64) -> None:
    if not chunk_records:
        print("No chunks to index.")
        return

    for start in range(0, len(chunk_records), batch_size):
        batch = chunk_records[start : start + batch_size]
        collection.add(
            ids=[r["id"] for r in batch],
            documents=[r["text"] for r in batch],
            metadatas=[
                {
                    "doc_id": r["doc_id"],
                    "company": r["company"],
                    "file_name": r["file_name"],
                    "source_type": r["source_type"],
                    "chunk_id": int(r["chunk_id"]),
                }
                for r in batch
            ],
        )
    print("Indexed", len(chunk_records), "chunks into ChromaDB.")


collection_count_before = collection.count()
print("Collection count before indexing:", collection_count_before)

if REBUILD_VECTOR_DB or collection_count_before == 0:
    index_chunks_to_chroma(collection, chunk_records)
else:
    print("Reusing existing collection. Skipping indexing because REBUILD_VECTOR_DB = False.")
    print("If We change chunking or embedding settings, We should use a new COLLECTION_NAME or set REBUILD_VECTOR_DB = True.")

print("Collection count:", collection.count())


## Step 4. Retrieval from the Vector DB (Hybrid + Table-First Rerank)

We use a hybrid retrieval strategy:
- semantic retrieval query (field meaning)
- table-oriented retrieval query (table headers + year/value patterns)
- rerank by table signals and numeric candidates

This improves recall for Scope 1/2/3 values that often live in dense tables.


In [ ]:
NUMBER_TOKEN_PATTERN = r"\b\d{1,3}(?:,\d{3})*(?:\.\d+)?\b|\b\d+\.\d+\b"


def retrieve_chunks(query: str, top_k: int = 4, filter_doc_id: str | None = None) -> List[Dict[str, Any]]:
    kwargs: Dict[str, Any] = {
        "query_texts": [query],
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"],
    }
    if filter_doc_id is not None:
        kwargs["where"] = {"doc_id": filter_doc_id}

    res = collection.query(**kwargs)

    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    distances = res.get("distances", [[]])[0]

    out = []
    for doc_text, meta, distance in zip(docs, metas, distances):
        item = dict(meta)
        item["text"] = doc_text
        item["distance"] = float(distance)
        # cosine distance -> rough similarity (teaching convenience)
        item["similarity"] = max(0.0, 1.0 - float(distance))
        out.append(item)
    return out


def retrieve_chunks_multiquery(
    query_texts: List[str],
    top_k_per_query: int = 8,
    filter_doc_id: str | None = None,
) -> List[Dict[str, Any]]:
    merged: Dict[str, Dict[str, Any]] = {}
    for q in query_texts:
        if not q.strip():
            continue
        hits = retrieve_chunks(q, top_k=top_k_per_query, filter_doc_id=filter_doc_id)
        for h in hits:
            key = f"{h['file_name']}::{h['chunk_id']}"
            prev = merged.get(key)
            item = dict(h)
            if prev is None:
                item["matched_queries"] = [q[:180]]
                merged[key] = item
            else:
                prev["similarity"] = max(prev["similarity"], item["similarity"])
                prev["distance"] = min(prev["distance"], item["distance"])
                prev.setdefault("matched_queries", [])
                if q[:180] not in prev["matched_queries"]:
                    prev["matched_queries"].append(q[:180])

    out = list(merged.values())
    out.sort(key=lambda x: x["similarity"], reverse=True)
    return out


def compute_table_signal_score(text: str) -> float:
    lower = text.lower()
    score = 0.0
    if re.search(r"\bscope\s*[123]\b", lower):
        score += 1.0
    if re.search(r"co2e|tco2|mtco2|metric tons?|mmt", lower):
        score += 1.0
    if re.search(r"data table|emissions category|gross emissions|totals|fiscal year|yoy", lower):
        score += 1.0
    if "2024" in lower and ("2023" in lower or "2022" in lower):
        score += 0.8
    number_count = len(re.findall(NUMBER_TOKEN_PATTERN, text))
    if number_count >= 6:
        score += 1.0
    elif number_count >= 3:
        score += 0.5
    return score


def looks_table_like(text: str) -> bool:
    return compute_table_signal_score(text) >= 2.0


example_query = "Scope 1 Scope 2 Scope 3 emissions total GHG emissions emissions intensity carbon targets reporting year"
if documents:
    example_doc_id = documents[0]["doc_id"]
    hits = retrieve_chunks_multiquery(
        query_texts=[
            example_query,
            "Data Tables Emissions Category 2024 Scope 1 Scope 2 Scope 3 tCO2e market-based",
        ],
        top_k_per_query=5,
        filter_doc_id=example_doc_id,
    )
    print("Retrieved chunks for:", example_doc_id)
    for i, h in enumerate(hits[:5], 1):
        print(
            f"\n[{i}] {h['file_name']} | chunk={h['chunk_id']} | "
            f"similarity={h['similarity']:.4f} | table_signal={compute_table_signal_score(h['text']):.2f}"
        )
        print(h["text"][:500])
else:
    print("No documents found.")


    ## Step 5. Use Colab Built-In AI Model (No API Key)

    We keep retrieval fixed and call Colab's built-in AI model for generation.

    Core interface (from the official Colab AI notebook):
    - `from google.colab import ai`
    - `ai.list_models()`
    - `ai.generate_text(prompt, model_name=...)`

    This keeps the teaching flow focused on RAG, not on API account setup.
    


In [ ]:
_COLAB_AI_CACHE: Dict[str, Any] = {}


def get_colab_ai_module():
    if "module" in _COLAB_AI_CACHE:
        return _COLAB_AI_CACHE["module"]

    if "google.colab" not in sys.modules:
        raise RuntimeError(
            "This notebook path uses google.colab.ai. "
            "Open the notebook in Google Colab to run the LLM step."
        )

    from google.colab import ai as colab_ai

    _COLAB_AI_CACHE["module"] = colab_ai
    return colab_ai


def _response_to_text(resp: Any) -> str:
    if isinstance(resp, str):
        return resp.strip()
    if hasattr(resp, "text") and isinstance(resp.text, str):
        return resp.text.strip()
    if isinstance(resp, dict):
        for key in ["text", "output_text", "response", "content"]:
            val = resp.get(key)
            if isinstance(val, str):
                return val.strip()
    return str(resp).strip()


def list_colab_ai_models(limit: int = 20):
    colab_ai = get_colab_ai_module()
    models = colab_ai.list_models()
    print("Available models (showing up to", limit, "):")
    try:
        for i, m in enumerate(models[:limit], 1):
            if isinstance(m, dict):
                name = m.get("name") or m.get("model_name") or str(m)
            else:
                name = getattr(m, "name", str(m))
            print(f"{i:02d}. {name}")
    except Exception:
        print(models)
    return models


def _is_model_unavailable_error(e: Exception) -> bool:
    msg = str(e).lower()
    return ("unavailable" in msg and "model" in msg) or "503" in msg


def _unique_keep_order(items: List[str]) -> List[str]:
    seen = set()
    out = []
    for x in items:
        if not x or x in seen:
            continue
        seen.add(x)
        out.append(x)
    return out


def call_colab_ai_llm(
    system_prompt: str,
    user_prompt: str,
    model_name: str | None = None,
) -> str:
    colab_ai = get_colab_ai_module()
    primary = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    fallback_models = list(globals().get("COLAB_AI_FALLBACK_MODELS", []))
    candidate_models = _unique_keep_order([primary, *fallback_models])

    combined_prompt = (
        "System instructions:\n"
        f"{system_prompt}\n\n"
        "User request:\n"
        f"{user_prompt}\n\n"
        "Return JSON only."
    )

    last_error = None
    _COLAB_AI_CACHE["last_model_used"] = None
    _COLAB_AI_CACHE["last_model_attempts"] = []

    for candidate in candidate_models:
        try:
            resp = colab_ai.generate_text(combined_prompt, model_name=candidate)
            _COLAB_AI_CACHE["last_model_used"] = candidate
            _COLAB_AI_CACHE["last_model_attempts"].append({"model": candidate, "status": "ok"})
            if candidate != primary:
                print(f"[Colab AI] Fallback activated: {primary} -> {candidate}")
            return _response_to_text(resp)
        except Exception as e:
            _COLAB_AI_CACHE["last_model_attempts"].append(
                {"model": candidate, "status": "error", "error": str(e)[:300]}
            )
            last_error = e
            if _is_model_unavailable_error(e):
                print(f"[Colab AI] Model unavailable: {candidate}. Trying next fallback...")
                continue
            raise

    if last_error is not None:
        raise last_error
    raise RuntimeError("No Colab AI models were configured for generation.")


def call_llm(system_prompt: str, user_prompt: str, model_name: str | None = None) -> str:
    return call_colab_ai_llm(system_prompt=system_prompt, user_prompt=user_prompt, model_name=model_name)


## Step 6. Expert Dictionaries + Field-Specific Prompts

We now use an expert extraction design:
- one expert dictionary per target field
- company-specific guidance for each field
- field-specific retrieval prompts
- HyDE-style hypothetical snippets
- table-first rerank with regex numeric candidates

This makes retrieval and extraction more robust for Scope 1/2/3 table-heavy disclosures.


In [ ]:
SYSTEM_PROMPT = """
You are an ESG reporting analyst specialized in GHG accounting.
We extract carbon metrics from report excerpts only.
We never invent values and we never use external assumptions.
If evidence is missing or ambiguous, we return null and explain in notes.
Return one JSON object only.
""".strip()

HYDE_SYSTEM_PROMPT = """
You write short hypothetical ESG report excerpts to improve retrieval.
Write in report language (tables, footnotes, methodology text).
Do not mention uncertainty, do not add commentary, and do not output JSON.
""".strip()

EXTRACTION_SCHEMA = {
    "company": "string or null",
    "report_year": "string or null",
    "scope_1_emissions": "string or null",
    "scope_2_emissions": "string or null",
    "scope_3_emissions": "string or null",
    "total_ghg_emissions": "string or null",
    "emissions_intensity": "string or null",
    "target_summary": "string or null",
    "evidence_quotes": ["short quotes/snippets from retrieved excerpts"],
    "confidence": "low | medium | high",
    "notes": "ambiguity / unit / boundary notes",
}

# Field-level expert dictionary: one extraction question + retrieval guidance per field.
FIELD_EXPERT_GUIDE: Dict[str, Dict[str, Any]] = {
    "report_year": {
        "question": "Which reporting year or period does the emissions data represent?",
        "definition": "Prefer data year / fiscal year of emissions inventory, not publication year.",
        "accepted_units": ["year", "fiscal year", "calendar year", "reporting period"],
        "retrieval_terms": [
            "reporting year",
            "fiscal year",
            "for the year ended",
            "inventory year",
            "baseline year",
        ],
        "disambiguation_rules": [
            "If both publication year and data year are present, prefer data year.",
            "If multiple years exist, choose the year tied to Scope totals.",
        ],
    },
    "scope_1_emissions": {
        "question": "What is Scope 1 GHG emissions value?",
        "definition": "Direct emissions from owned or controlled sources.",
        "accepted_units": ["tCO2e", "mtCO2e", "metric tons CO2e", "million metric tons CO2e"],
        "retrieval_terms": [
            "Scope 1",
            "direct emissions",
            "stationary combustion",
            "fleet fuel",
            "process emissions",
        ],
        "disambiguation_rules": [
            "Keep reported unit exactly as written.",
            "Do not mix Scope 1 with Scope 1+2 totals unless explicit.",
        ],
    },
    "scope_2_emissions": {
        "question": "What is Scope 2 GHG emissions value?",
        "definition": "Indirect emissions from purchased electricity/steam/heat/cooling.",
        "accepted_units": ["tCO2e", "mtCO2e", "location-based", "market-based"],
        "retrieval_terms": [
            "Scope 2",
            "purchased electricity",
            "location-based",
            "market-based",
            "residual mix",
        ],
        "disambiguation_rules": [
            "If both location-based and market-based exist, record what is clearly labeled.",
            "Use notes to explain if both values appear.",
        ],
    },
    "scope_3_emissions": {
        "question": "What is Scope 3 GHG emissions value?",
        "definition": "Other indirect value-chain emissions (upstream and downstream categories).",
        "accepted_units": ["tCO2e", "mtCO2e", "category-level scope 3 values"],
        "retrieval_terms": [
            "Scope 3",
            "value chain emissions",
            "upstream",
            "downstream",
            "category",
        ],
        "disambiguation_rules": [
            "Prefer total Scope 3 value when category details also appear.",
            "If partial category coverage is reported, explain in notes.",
        ],
    },
    "total_ghg_emissions": {
        "question": "What is total GHG emissions value?",
        "definition": "Company total emissions in the report boundary (often Scope 1+2+3 or operational total).",
        "accepted_units": ["tCO2e", "mtCO2e", "total emissions"],
        "retrieval_terms": [
            "total GHG emissions",
            "gross emissions",
            "absolute emissions",
            "combined scope",
            "total carbon footprint",
        ],
        "disambiguation_rules": [
            "Do not calculate total unless explicitly reported.",
            "Capture boundary notes if total excludes categories.",
        ],
    },
    "emissions_intensity": {
        "question": "What emissions intensity metric is reported?",
        "definition": "Emission intensity normalized by output, revenue, user, square-foot, or other denominator.",
        "accepted_units": ["tCO2e/revenue", "gCO2e/unit", "kgCO2e/product", "intensity index"],
        "retrieval_terms": [
            "emissions intensity",
            "carbon intensity",
            "intensity metric",
            "per unit",
            "per revenue",
            "normalized emissions",
        ],
        "disambiguation_rules": [
            "Keep numerator and denominator exactly as written.",
            "If multiple intensity metrics exist, choose the primary one and note others.",
        ],
    },
    "target_summary": {
        "question": "What key emissions reduction target(s) are stated?",
        "definition": "Near-term and long-term decarbonization targets with baseline year and target year.",
        "accepted_units": ["%", "net-zero year", "absolute reduction target", "intensity target"],
        "retrieval_terms": [
            "target",
            "goal",
            "net zero",
            "science based target",
            "SBTi",
            "by 2030",
            "by 2040",
            "by 2050",
        ],
        "disambiguation_rules": [
            "Prefer explicit target statements with year and baseline.",
            "If multiple targets exist, summarize the most material carbon target(s).",
        ],
    },
}

# Company-specific guidance dictionary: each company has targeted guidance for each extraction field.
COMPANY_FIELD_GUIDANCE: Dict[str, Dict[str, str]] = {
    "Amazon": {
        "report_year": "Amazon frequently labels values by reporting year in sustainability data tables. We use that table year.",
        "scope_1_emissions": "Amazon may include transportation and facilities in Scope 1 discussion. We look for explicit Scope 1 totals.",
        "scope_2_emissions": "Amazon often provides market-based values with renewable claims. We record the clearly labeled Scope 2 metric.",
        "scope_3_emissions": "Amazon Scope 3 is usually material and category-heavy. We prioritize the headline Scope 3 total.",
        "total_ghg_emissions": "Amazon may report gross total emissions under a corporate accounting boundary. We capture that reported total only.",
        "emissions_intensity": "Amazon can report intensity versus gross merchandise sales or other business metrics. We preserve exact denominator.",
        "target_summary": "Amazon targets often include net-zero and interim milestones. We capture target year, baseline, and reduction statement.",
    },
    "Google": {
        "report_year": "Google environmental reports can publish in one year while describing prior-year inventory data.",
        "scope_1_emissions": "Google Scope 1 may be comparatively smaller and tied to fuel/process sources. We use explicit inventory values.",
        "scope_2_emissions": "Google may discuss 24/7 CFE and market instruments. We distinguish Scope 2 accounting value from energy strategy text.",
        "scope_3_emissions": "Google Scope 3 can include hardware supply chain and use-phase context. We extract only explicit Scope 3 totals.",
        "total_ghg_emissions": "Google may show absolute emissions and operational subsets. We keep the exact boundary label in notes.",
        "emissions_intensity": "Google may express intensity using computational or business growth frames. We retain exact metric expression.",
        "target_summary": "Google targets may include net-zero operations and value-chain goals. We summarize explicit measurable commitments.",
    },
    "Apple": {
        "report_year": "Apple environmental progress reports may reference fiscal-year emissions and product-year context together.",
        "scope_1_emissions": "Apple Scope 1 often appears in corporate operations inventory tables. We prioritize explicit Scope 1 values.",
        "scope_2_emissions": "Apple may emphasize renewable electricity progress. We ensure we capture official Scope 2 accounting value.",
        "scope_3_emissions": "Apple Scope 3 is typically dominant and supply-chain focused. We capture reported Scope 3 totals when explicit.",
        "total_ghg_emissions": "Apple may report product lifecycle and corporate totals separately. We extract the explicit total under stated boundary.",
        "emissions_intensity": "Apple may report intensity by product or value-chain framing. We preserve metric formula wording.",
        "target_summary": "Apple often states carbon neutrality roadmap targets by year. We summarize concrete target statements.",
    },
    "Meta": {
        "report_year": "Meta sustainability reporting may label inventory by data year and publish later. We choose data year.",
        "scope_1_emissions": "Meta Scope 1 may involve on-site fuel and backup systems. We extract explicit Scope 1 values only.",
        "scope_2_emissions": "Meta may emphasize renewable matching and market-based reporting. We track the labeled Scope 2 basis.",
        "scope_3_emissions": "Meta Scope 3 often includes supply chain and capital goods impacts. We capture the reported total Scope 3 value.",
        "total_ghg_emissions": "Meta may provide total emissions and separate operational subsets. We keep the directly reported total.",
        "emissions_intensity": "Meta intensity can be linked to revenue, user activity, or operations. We keep denominator exact.",
        "target_summary": "Meta targets can include net-zero value-chain timelines. We summarize baseline year + target year + reduction claim.",
    },
}


def get_field_spec(field_key: str) -> Dict[str, Any]:
    return FIELD_EXPERT_GUIDE[field_key]


def get_company_guidance(company_name: str, field_key: str) -> str:
    return COMPANY_FIELD_GUIDANCE.get(company_name, {}).get(
        field_key,
        "We use explicit labels and preserve original units and boundaries.",
    )


def build_hyde_prompt(company_name: str, field_key: str) -> str:
    spec = get_field_spec(field_key)
    company_hint = get_company_guidance(company_name, field_key)
    retrieval_terms = ", ".join(spec["retrieval_terms"])
    accepted_units = ", ".join(spec["accepted_units"])
    return f"""
Company: {company_name}
Field: {field_key}
Field question: {spec['question']}
Definition: {spec['definition']}
Likely retrieval terms: {retrieval_terms}
Likely unit patterns: {accepted_units}
Company-specific guidance: {company_hint}

Write 3 short hypothetical report-style lines that could appear in an ESG report and help vector retrieval.
Use factual report tone, include likely table labels and boundary language.
Do not output JSON.
""".strip()


def build_field_retrieval_query(company_name: str, field_key: str, hyde_text: str = "") -> str:
    spec = get_field_spec(field_key)
    company_hint = get_company_guidance(company_name, field_key)
    retrieval_terms = " ".join(spec["retrieval_terms"])
    rules = " ".join(spec["disambiguation_rules"])
    query = f"""
Retrieve evidence for company {company_name}.
Target field: {field_key}
Question: {spec['question']}
Definition: {spec['definition']}
Important terms: {retrieval_terms}
Disambiguation rules: {rules}
Company guidance: {company_hint}
""".strip()
    if hyde_text:
        query += "\n\nHypothetical report wording for retrieval expansion:\n" + hyde_text
    return query


def format_retrieved_chunks(chunks: List[Dict[str, Any]], max_chars_per_chunk: int = 650) -> str:
    if not chunks:
        return "<no chunks>"
    blocks = []
    for i, c in enumerate(chunks, 1):
        blocks.append(
            f"[Chunk {i}] file={c['file_name']} chunk_id={c['chunk_id']} similarity={c['similarity']:.4f}\n"
            f"{c['text'][:max_chars_per_chunk]}"
        )
    return "\n\n".join(blocks)


def build_extraction_prompt(
    company_name: str,
    retrieval_package: Dict[str, Any],
) -> str:
    field_blocks = []
    for field_key in FIELD_EXPERT_GUIDE.keys():
        spec = get_field_spec(field_key)
        field_data = retrieval_package["by_field"].get(field_key, {})
        company_hint = get_company_guidance(company_name, field_key)
        field_chunks = field_data.get("hits", [])

        field_blocks.append(
            f"""
Field: {field_key}
Question: {spec['question']}
Definition: {spec['definition']}
Company-specific extraction guidance: {company_hint}
Accepted unit patterns: {", ".join(spec['accepted_units'])}
Disambiguation reminders: {" | ".join(spec['disambiguation_rules'])}

Retrieved excerpts for this field:
{format_retrieved_chunks(field_chunks)}
""".strip()
        )

    merged_chunks_block = format_retrieved_chunks(retrieval_package.get("merged_hits", []), max_chars_per_chunk=700)

    return f"""
Task: Extract carbon emissions information for {company_name}.

Output JSON schema (use these exact keys):
{json.dumps(EXTRACTION_SCHEMA, indent=2)}

Field-by-field expert extraction guidance:
{chr(10).join([''] + [b + chr(10) for b in field_blocks])}

Cross-field merged evidence pool:
{merged_chunks_block}

Output rules:
1. Use only retrieved excerpts in this prompt.
2. Keep values exactly as written (including unit and scale markers like million).
3. Do not infer missing numbers from other fields.
4. If two valid values exist (for example location-based and market-based), pick the one most clearly tied to the requested field and explain in notes.
5. evidence_quotes must include direct short snippets that support key fields.
6. Return one valid JSON object only (no markdown, no extra text).
""".strip()


# --- Table-first retrieval helpers and regex candidate extraction ---
FIELD_REGEX_RULES: Dict[str, Dict[str, Any]] = {
    "report_year": {
        "keywords": ["reporting year", "fiscal year", "year ended", "inventory year"],
        "unit_regex": r"year|fy\d{2,4}",
    },
    "scope_1_emissions": {
        "keywords": ["scope 1", "direct operations", "direct emissions"],
        "unit_regex": r"tco2e|mtco2e|co2e|metric tons?|mmt",
    },
    "scope_2_emissions": {
        "keywords": ["scope 2", "purchased electricity", "market-based", "location-based"],
        "unit_regex": r"tco2e|mtco2e|co2e|metric tons?|mmt|market-based|location-based",
    },
    "scope_3_emissions": {
        "keywords": ["scope 3", "indirect sources", "value chain", "category"],
        "unit_regex": r"tco2e|mtco2e|co2e|metric tons?|mmt",
    },
    "total_ghg_emissions": {
        "keywords": ["total ghg", "gross emissions", "absolute emissions", "carbon footprint"],
        "unit_regex": r"tco2e|mtco2e|co2e|metric tons?|mmt",
    },
    "emissions_intensity": {
        "keywords": ["intensity", "per ", "gco2e", "kgco2e"],
        "unit_regex": r"gco2e|kgco2e|tco2e|per\s+\$|per\s+unit|per\s+vehicle|per\s+revenue",
    },
    "target_summary": {
        "keywords": ["target", "goal", "net zero", "science based", "sbti"],
        "unit_regex": r"%|2030|2040|2050|net\s*zero|science\s*based",
    },
}


def build_field_table_query(company_name: str, field_key: str) -> str:
    spec = get_field_spec(field_key)
    retrieval_terms = " ".join(spec["retrieval_terms"])
    return f"""
Company {company_name} ESG data table for {field_key}.
Find rows/columns that look like:
- Data Tables / Emissions Category / Gross emissions / Totals
- Years like 2024 2023 2022
- Units like tCO2e, mtCO2e, metric tons CO2e, market-based, location-based
Focus terms: {retrieval_terms}
""".strip()


def _candidate_segments(text: str) -> List[str]:
    parts = re.split(r"\n+|(?<=[.;])\s+", text)
    out = []
    for p in parts:
        p = p.strip()
        if p and len(p) >= 12:
            out.append(p)
    return out


def extract_field_numeric_candidates(field_key: str, text: str, max_candidates: int = 6) -> List[Dict[str, str]]:
    rule = FIELD_REGEX_RULES.get(field_key, {})
    keywords = [k.lower() for k in rule.get("keywords", [])]
    unit_regex = rule.get("unit_regex", r"")
    candidates: List[Dict[str, str]] = []

    for seg in _candidate_segments(text):
        seg_lower = seg.lower()
        keyword_hit = any(k in seg_lower for k in keywords) if keywords else True
        if not keyword_hit:
            continue

        nums = re.findall(NUMBER_TOKEN_PATTERN, seg)
        if not nums:
            continue

        unit_match = re.search(unit_regex, seg_lower) if unit_regex else None
        candidates.append(
            {
                "line": seg[:240],
                "numbers": ", ".join(nums[:6]),
                "unit_hint": unit_match.group(0) if unit_match else "",
            }
        )
        if len(candidates) >= max_candidates:
            break

    return candidates


def dedupe_regex_candidates(items: List[Dict[str, str]], max_items: int = 8) -> List[Dict[str, str]]:
    out = []
    seen = set()
    for item in items:
        key = (item.get("numbers", ""), item.get("unit_hint", ""), item.get("line", "")[:120])
        if key in seen:
            continue
        seen.add(key)
        out.append(item)
        if len(out) >= max_items:
            break
    return out


def format_regex_candidates(candidates: List[Dict[str, str]], max_items: int = 6) -> str:
    if not candidates:
        return "<no numeric candidates>"
    lines = []
    for i, c in enumerate(candidates[:max_items], 1):
        unit = f" | unit_hint={c.get('unit_hint')}" if c.get("unit_hint") else ""
        lines.append(f"{i}. numbers={c.get('numbers','')}{unit} | line={c.get('line','')[:180]}")
    return "\n".join(lines)


def format_retrieved_chunks(chunks: List[Dict[str, Any]], max_chars_per_chunk: int = 650) -> str:
    if not chunks:
        return "<no chunks>"
    blocks = []
    for i, c in enumerate(chunks, 1):
        rerank = c.get("rerank_score")
        table_score = c.get("table_signal_score")
        rerank_text = f"{rerank:.4f}" if isinstance(rerank, (int, float)) else "n/a"
        table_text = f"{table_score:.2f}" if isinstance(table_score, (int, float)) else "n/a"
        blocks.append(
            f"[Chunk {i}] file={c['file_name']} chunk_id={c['chunk_id']} "
            f"similarity={c['similarity']:.4f} rerank={rerank_text} table_signal={table_text}\n"
            f"{c['text'][:max_chars_per_chunk]}"
        )
    return "\n\n".join(blocks)


def build_extraction_prompt(
    company_name: str,
    retrieval_package: Dict[str, Any],
) -> str:
    field_blocks = []
    for field_key in FIELD_EXPERT_GUIDE.keys():
        spec = get_field_spec(field_key)
        field_data = retrieval_package["by_field"].get(field_key, {})
        company_hint = get_company_guidance(company_name, field_key)
        field_chunks = field_data.get("hits", [])
        regex_candidates = field_data.get("regex_candidates", [])

        field_blocks.append(
            f"""
Field: {field_key}
Question: {spec['question']}
Definition: {spec['definition']}
Company-specific extraction guidance: {company_hint}
Accepted unit patterns: {", ".join(spec['accepted_units'])}
Disambiguation reminders: {" | ".join(spec['disambiguation_rules'])}

Regex numeric candidates (table-first pre-extraction):
{format_regex_candidates(regex_candidates)}

Retrieved excerpts for this field:
{format_retrieved_chunks(field_chunks)}
""".strip()
        )

    merged_chunks_block = format_retrieved_chunks(retrieval_package.get("merged_hits", []), max_chars_per_chunk=700)

    return f"""
Task: Extract carbon emissions information for {company_name}.

Output JSON schema (use these exact keys):
{json.dumps(EXTRACTION_SCHEMA, indent=2)}

Field-by-field expert extraction guidance:
{chr(10).join([''] + [b + chr(10) for b in field_blocks])}

Cross-field merged evidence pool:
{merged_chunks_block}

Output rules:
1. Use only retrieved excerpts and regex candidates in this prompt.
2. Keep values exactly as written (including unit and scale markers like million).
3. Do not infer missing numbers from other fields.
4. If two valid values exist (for example location-based and market-based), pick the one most clearly tied to the requested field and explain in notes.
5. For Scope 1/2/3, prefer explicit table rows with units.
6. evidence_quotes must include direct short snippets that support key fields.
7. Return one valid JSON object only (no markdown, no extra text).
""".strip()


In [ ]:
def extract_first_json_object(text: str) -> str:
    text = text.strip()
    # Fast path
    if text.startswith("{") and text.endswith("}"):
        return text

    # Find first balanced JSON object
    start = text.find("{")
    if start == -1:
        raise ValueError("No JSON object start found")

    depth = 0
    for i in range(start, len(text)):
        ch = text[i]
        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start : i + 1]
    raise ValueError("No balanced JSON object found")


def parse_model_json(raw_text: str) -> Dict[str, Any]:
    try:
        return json.loads(raw_text)
    except Exception:
        candidate = extract_first_json_object(raw_text)
        return json.loads(candidate)


## Step 7. Field-Aware Retrieval + Table-First Rerank + Extraction

This function now combines:
- field-specific retrieval prompts
- company-specific guidance per field
- HyDE-style hypothetical snippets
- table-oriented retrieval query
- regex numeric candidate mining
- rerank before final extraction


In [ ]:
RAG_FIELDS = list(FIELD_EXPERT_GUIDE.keys())
HYDE_CACHE: Dict[str, str] = {}


def generate_hyde_text(company_name: str, field_key: str, hyde_model_name: str | None = None) -> str:
    model_name = hyde_model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    cache_key = f"{company_name}::{field_key}::{model_name}"
    if cache_key in HYDE_CACHE:
        return HYDE_CACHE[cache_key]

    user_prompt = build_hyde_prompt(company_name, field_key)
    hyde_text = call_llm(system_prompt=HYDE_SYSTEM_PROMPT, user_prompt=user_prompt, model_name=model_name)
    hyde_text = re.sub(r"\s+", " ", hyde_text).strip()
    HYDE_CACHE[cache_key] = hyde_text
    return hyde_text


def rerank_hits_for_field(
    field_key: str,
    hits: List[Dict[str, Any]],
    table_signal_weight: float = 0.08,
    regex_candidate_weight: float = 0.03,
) -> List[Dict[str, Any]]:
    ranked: List[Dict[str, Any]] = []
    for h in hits:
        item = dict(h)
        text = item.get("text", "")
        table_signal_score = compute_table_signal_score(text)
        regex_candidates = extract_field_numeric_candidates(field_key, text, max_candidates=6)

        rerank_score = float(item.get("similarity", 0.0))
        rerank_score += table_signal_weight * table_signal_score
        rerank_score += regex_candidate_weight * min(len(regex_candidates), 4)

        item["table_signal_score"] = round(table_signal_score, 3)
        item["regex_candidates"] = regex_candidates
        item["rerank_score"] = round(rerank_score, 6)
        ranked.append(item)

    ranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return ranked


def retrieve_field_aware_context(
    doc: Dict[str, Any],
    top_k_per_field: int = 4,
    retrieval_pool_size_per_field: int = 12,
    use_hyde_retrieval: bool = True,
    hyde_model_name: str | None = None,
    use_table_query_retrieval: bool = True,
    table_signal_weight: float = 0.08,
    regex_candidate_weight: float = 0.03,
) -> Dict[str, Any]:
    by_field: Dict[str, Dict[str, Any]] = {}
    merged_map: Dict[str, Dict[str, Any]] = {}
    hyde_errors: Dict[str, str] = {}

    for field_key in RAG_FIELDS:
        hyde_text = ""
        if use_hyde_retrieval:
            try:
                hyde_text = generate_hyde_text(doc["company"], field_key, hyde_model_name=hyde_model_name)
            except Exception as e:
                hyde_errors[field_key] = str(e)

        semantic_query = build_field_retrieval_query(doc["company"], field_key, hyde_text=hyde_text)
        table_query = build_field_table_query(doc["company"], field_key)

        query_texts = [semantic_query]
        if use_table_query_retrieval:
            query_texts.append(table_query)

        pooled_hits = retrieve_chunks_multiquery(
            query_texts=query_texts,
            top_k_per_query=retrieval_pool_size_per_field,
            filter_doc_id=doc["doc_id"],
        )

        ranked_hits = rerank_hits_for_field(
            field_key=field_key,
            hits=pooled_hits,
            table_signal_weight=table_signal_weight,
            regex_candidate_weight=regex_candidate_weight,
        )

        final_hits = ranked_hits[:top_k_per_field]

        all_candidates: List[Dict[str, str]] = []
        for h in final_hits:
            all_candidates.extend(h.get("regex_candidates", []))
        dedup_candidates = dedupe_regex_candidates(all_candidates, max_items=10)

        by_field[field_key] = {
            "query_texts": query_texts,
            "hyde_text": hyde_text,
            "hits": final_hits,
            "regex_candidates": dedup_candidates,
        }

        for h in final_hits:
            key = f"{h['file_name']}::{h['chunk_id']}"
            prev = merged_map.get(key)
            if prev is None or h["rerank_score"] > prev.get("rerank_score", 0.0):
                merged_map[key] = h

    merged_hits = sorted(merged_map.values(), key=lambda x: x.get("rerank_score", x["similarity"]), reverse=True)
    return {
        "by_field": by_field,
        "merged_hits": merged_hits,
        "hyde_errors": hyde_errors,
    }


def rag_extract_for_doc(
    doc: Dict[str, Any],
    model_name: str | None = None,
    top_k_per_field: int = 4,
    retrieval_pool_size_per_field: int = 12,
    use_hyde_retrieval: bool = True,
    hyde_model_name: str | None = None,
    use_table_query_retrieval: bool = True,
    table_signal_weight: float = 0.08,
    regex_candidate_weight: float = 0.03,
) -> Dict[str, Any]:
    retrieval_package = retrieve_field_aware_context(
        doc,
        top_k_per_field=top_k_per_field,
        retrieval_pool_size_per_field=retrieval_pool_size_per_field,
        use_hyde_retrieval=use_hyde_retrieval,
        hyde_model_name=hyde_model_name,
        use_table_query_retrieval=use_table_query_retrieval,
        table_signal_weight=table_signal_weight,
        regex_candidate_weight=regex_candidate_weight,
    )

    merged_hits = retrieval_package["merged_hits"]
    if not merged_hits:
        return {
            "company": doc["company"],
            "error": f"No retrieved evidence for {doc['doc_id']}",
            "_rag_meta": {
                "provider": "colab_ai",
                "doc_id": doc["doc_id"],
                "file_name": doc["file_name"],
                "source_type": doc["source_type"],
            },
        }

    user_prompt = build_extraction_prompt(doc["company"], retrieval_package)
    t0 = time.time()
    raw = call_llm(system_prompt=SYSTEM_PROMPT, user_prompt=user_prompt, model_name=model_name)
    elapsed = time.time() - t0

    try:
        data = parse_model_json(raw)
    except Exception as e:
        data = {
            "company": doc["company"],
            "parse_error": str(e),
            "raw_model_output": raw[:7000],
        }

    requested_model = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    actual_model = _COLAB_AI_CACHE.get("last_model_used") or requested_model
    field_hit_counts = {k: len(v.get("hits", [])) for k, v in retrieval_package["by_field"].items()}
    field_candidate_counts = {k: len(v.get("regex_candidates", [])) for k, v in retrieval_package["by_field"].items()}

    data["_rag_meta"] = {
        "provider": "colab_ai",
        "model": actual_model,
        "requested_model": requested_model,
        "model_attempts": _COLAB_AI_CACHE.get("last_model_attempts", []),
        "doc_id": doc["doc_id"],
        "file_name": doc["file_name"],
        "source_type": doc["source_type"],
        "top_k_per_field": top_k_per_field,
        "retrieval_pool_size_per_field": retrieval_pool_size_per_field,
        "use_hyde_retrieval": use_hyde_retrieval,
        "hyde_model_name": hyde_model_name or requested_model,
        "use_table_query_retrieval": use_table_query_retrieval,
        "table_signal_weight": table_signal_weight,
        "regex_candidate_weight": regex_candidate_weight,
        "hyde_error_fields": list(retrieval_package.get("hyde_errors", {}).keys()),
        "merged_hit_count": len(merged_hits),
        "field_hit_counts": field_hit_counts,
        "field_candidate_counts": field_candidate_counts,
        "latency_sec": round(elapsed, 2),
        "retrieved_chunks": [
            {
                "chunk_id": h["chunk_id"],
                "similarity": h["similarity"],
                "rerank_score": h.get("rerank_score"),
                "table_signal_score": h.get("table_signal_score"),
                "file_name": h["file_name"],
            }
            for h in merged_hits[:14]
        ],
    }
    return data


## Step 8. Configure Model + Retrieval Strategy

We configure:
- extraction model + fallbacks
- field-level retrieval depth
- HyDE expansion
- table-first rerank weights


In [ ]:
# Preferred Colab built-in AI model (quality-first for final extraction)
COLAB_AI_MODEL_NAME = "google/gemini-2.5-pro"

# Automatic fallback order for temporary model unavailability (503 / unavailable)
COLAB_AI_FALLBACK_MODELS = [
    "google/gemini-2.5-flash",
    "google/gemini-2.0-flash",
    "google/gemini-2.5-flash-lite",
    "google/gemini-2.0-flash-lite",
]

# Retrieval strategy controls
TOP_K_PER_FIELD = 4
RETRIEVAL_POOL_SIZE_PER_FIELD = 12
USE_HYDE_RETRIEVAL = True
HYDE_MODEL_NAME = "google/gemini-2.5-flash-lite"
USE_TABLE_QUERY_RETRIEVAL = True
TABLE_SIGNAL_WEIGHT = 0.08
REGEX_CANDIDATE_WEIGHT = 0.03

# Optional diagnostics
LIST_AVAILABLE_COLAB_MODELS = False

os.environ["COLAB_AI_MODEL_NAME"] = COLAB_AI_MODEL_NAME

print("LLM provider = colab_ai (built-in, no API key)")
print("Extraction model =", COLAB_AI_MODEL_NAME)
print("Fallback models =", COLAB_AI_FALLBACK_MODELS)
print("TOP_K_PER_FIELD =", TOP_K_PER_FIELD)
print("RETRIEVAL_POOL_SIZE_PER_FIELD =", RETRIEVAL_POOL_SIZE_PER_FIELD)
print("USE_HYDE_RETRIEVAL =", USE_HYDE_RETRIEVAL)
print("HYDE_MODEL_NAME =", HYDE_MODEL_NAME)
print("USE_TABLE_QUERY_RETRIEVAL =", USE_TABLE_QUERY_RETRIEVAL)
print("TABLE_SIGNAL_WEIGHT =", TABLE_SIGNAL_WEIGHT)
print("REGEX_CANDIDATE_WEIGHT =", REGEX_CANDIDATE_WEIGHT)
print("documents =", [d["company"] for d in documents])

if LIST_AVAILABLE_COLAB_MODELS:
    try:
        list_colab_ai_models(limit=30)
    except Exception as e:
        print("Could not list models:", e)


In [ ]:
RUN_COLAB_AI_SMOKE_TEST = False

if not IN_COLAB:
    print("Local environment detected. The LLM step will not run here because google.colab.ai is Colab-only.")
else:
    try:
        _ = get_colab_ai_module()
        print("google.colab.ai is ready.")
        print("Preferred model =", os.getenv("COLAB_AI_MODEL_NAME", "(unset)"))
        print("Fallback models =", globals().get("COLAB_AI_FALLBACK_MODELS", []))
        if RUN_COLAB_AI_SMOKE_TEST:
            test_resp = call_colab_ai_llm(
                system_prompt="Reply with plain text only.",
                user_prompt="Say OK in one word.",
                model_name=os.getenv("COLAB_AI_MODEL_NAME"),
            )
            print("Smoke test response:", test_resp)
            print("Actual model used =", _COLAB_AI_CACHE.get("last_model_used"))
        else:
            print("Set RUN_COLAB_AI_SMOKE_TEST = True if We want a quick one-line generation test.")
    except Exception as e:
        print("Colab AI check failed:", e)


    ## Step 9. Test One Company First (Recommended)

    We run one company first to confirm retrieval + prompting + Colab AI extraction works before batch mode.
    


In [ ]:
RUN_ONE_COMPANY = bool(IN_COLAB)  # local runtime cannot call google.colab.ai

single_result = None
if RUN_ONE_COMPANY and documents:
    preferred_order = ["Amazon", "Google", "Apple", "Meta"]
    selected = None
    for name in preferred_order:
        selected = next((d for d in documents if d["company"] == name), None)
        if selected:
            break

    print("Running one-company run for:", selected["company"], "|", selected["file_name"])
    single_result = rag_extract_for_doc(
        selected,
        model_name=COLAB_AI_MODEL_NAME,
        top_k_per_field=TOP_K_PER_FIELD,
        retrieval_pool_size_per_field=RETRIEVAL_POOL_SIZE_PER_FIELD,
        use_hyde_retrieval=USE_HYDE_RETRIEVAL,
        hyde_model_name=HYDE_MODEL_NAME,
        use_table_query_retrieval=USE_TABLE_QUERY_RETRIEVAL,
        table_signal_weight=TABLE_SIGNAL_WEIGHT,
        regex_candidate_weight=REGEX_CANDIDATE_WEIGHT,
    )
    print(json.dumps(single_result, indent=2, ensure_ascii=False)[:5000])
else:
    print("Skipping one-company run (not in Colab or no documents).")


    ## Step 10. Batch Run on Amazon / Google / Apple / Meta

    We run the same RAG pipeline across the four-company set and save outputs.
    


In [ ]:
TARGET_COMPANIES = ["Amazon", "Google", "Apple", "Meta"]
RUN_BATCH = bool(IN_COLAB)  # local runtime cannot call google.colab.ai


def batch_extract(
    docs: List[Dict[str, Any]],
    model_name: str | None = None,
    top_k_per_field: int = 4,
    retrieval_pool_size_per_field: int = 12,
    use_hyde_retrieval: bool = True,
    hyde_model_name: str | None = None,
    use_table_query_retrieval: bool = True,
    table_signal_weight: float = 0.08,
    regex_candidate_weight: float = 0.03,
    target_companies: List[str] | None = None,
) -> List[Dict[str, Any]]:
    selected_docs = docs
    if target_companies:
        target_set = set(target_companies)
        selected_docs = [d for d in docs if d["company"] in target_set]

    results = []
    for doc in selected_docs:
        print(f"\n=== Processing {doc['company']} | {doc['file_name']} ===")
        try:
            result = rag_extract_for_doc(
                doc,
                model_name=model_name,
                top_k_per_field=top_k_per_field,
                retrieval_pool_size_per_field=retrieval_pool_size_per_field,
                use_hyde_retrieval=use_hyde_retrieval,
                hyde_model_name=hyde_model_name,
                use_table_query_retrieval=use_table_query_retrieval,
                table_signal_weight=table_signal_weight,
                regex_candidate_weight=regex_candidate_weight,
            )
        except Exception as e:
            result = {
                "company": doc["company"],
                "error": str(e),
                "_rag_meta": {
                    "provider": "colab_ai",
                    "model": model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro"),
                    "doc_id": doc["doc_id"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                },
            }
        results.append(result)
    return results


all_results: List[Dict[str, Any]] = []
if RUN_BATCH:
    all_results = batch_extract(
        documents,
        model_name=COLAB_AI_MODEL_NAME,
        top_k_per_field=TOP_K_PER_FIELD,
        retrieval_pool_size_per_field=RETRIEVAL_POOL_SIZE_PER_FIELD,
        use_hyde_retrieval=USE_HYDE_RETRIEVAL,
        hyde_model_name=HYDE_MODEL_NAME,
        use_table_query_retrieval=USE_TABLE_QUERY_RETRIEVAL,
        table_signal_weight=TABLE_SIGNAL_WEIGHT,
        regex_candidate_weight=REGEX_CANDIDATE_WEIGHT,
        target_companies=TARGET_COMPANIES,
    )
    print("\nBatch complete:", len(all_results), "result(s)")
else:
    print("RUN_BATCH = False (set IN_COLAB runtime to run end-to-end extraction).")


## Step 11. Convert Results to a Table

This makes the extraction output easier to inspect and compare across firms.


In [ ]:
summary_df = pd.DataFrame(
    [
        {
            "company": r.get("company"),
            "report_year": r.get("report_year"),
            "scope_1_emissions": r.get("scope_1_emissions"),
            "scope_2_emissions": r.get("scope_2_emissions"),
            "scope_3_emissions": r.get("scope_3_emissions"),
            "total_ghg_emissions": r.get("total_ghg_emissions"),
            "emissions_intensity": r.get("emissions_intensity"),
            "target_summary": r.get("target_summary"),
            "confidence": r.get("confidence"),
            "notes": r.get("notes"),
            "error": r.get("error"),
            "parse_error": r.get("parse_error"),
            "latency_sec": (r.get("_rag_meta") or {}).get("latency_sec"),
            "merged_hit_count": (r.get("_rag_meta") or {}).get("merged_hit_count"),
            "use_hyde_retrieval": (r.get("_rag_meta") or {}).get("use_hyde_retrieval"),
            "use_table_query_retrieval": (r.get("_rag_meta") or {}).get("use_table_query_retrieval"),
            "source_type": (r.get("_rag_meta") or {}).get("source_type"),
            "file_name": (r.get("_rag_meta") or {}).get("file_name"),
        }
        for r in all_results
    ]
)

summary_df


    ## Step 12. Save Outputs and Run Record

    We save:
    - JSON results
    - CSV summary table
    - a run-record JSON (Colab AI model, timestamp, files used)
    


In [ ]:
run_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

results_json_path = OUTPUT_DIR / "esg_carbon_rag_results.json"
summary_csv_path = OUTPUT_DIR / "esg_carbon_rag_results.csv"
run_record_path = OUTPUT_DIR / f"notebook_run_record_{run_timestamp}.json"

with open(results_json_path, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=2, ensure_ascii=False)

summary_df.to_csv(summary_csv_path, index=False)

run_record = {
    "run_timestamp_utc": run_timestamp,
    "provider": "colab_ai",
    "model": COLAB_AI_MODEL_NAME,
    "model_fallbacks": COLAB_AI_FALLBACK_MODELS,
    "top_k_per_field": TOP_K_PER_FIELD,
    "retrieval_pool_size_per_field": RETRIEVAL_POOL_SIZE_PER_FIELD,
    "use_hyde_retrieval": USE_HYDE_RETRIEVAL,
    "hyde_model_name": HYDE_MODEL_NAME,
    "use_table_query_retrieval": USE_TABLE_QUERY_RETRIEVAL,
    "table_signal_weight": TABLE_SIGNAL_WEIGHT,
    "regex_candidate_weight": REGEX_CANDIDATE_WEIGHT,
    "target_companies": TARGET_COMPANIES,
    "doc_count_loaded": len(documents),
    "doc_files": [d["file_name"] for d in documents],
    "vector_db_path": str(VECTOR_DB_DIR),
    "vector_collection": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "embedding_device": EMBEDDING_DEVICE,
    "rebuild_vector_db": REBUILD_VECTOR_DB,
    "chunk_max_chars": CHUNK_MAX_CHARS,
    "chunk_overlap_chars": CHUNK_OVERLAP_CHARS,
    "results_json_path": str(results_json_path),
    "summary_csv_path": str(summary_csv_path),
    "uses_colab_builtin_ai": True,
}

with open(run_record_path, "w", encoding="utf-8") as f:
    json.dump(run_record, f, indent=2, ensure_ascii=False)

print("Saved:", results_json_path)
print("Saved:", summary_csv_path)
print("Saved:", run_record_path)


## Step 13. Climate Commitment Scoring (TCFD-Aligned)

When exact emissions numbers are difficult to extract, We can still evaluate climate-commitment quality.

We score each company on five TCFD-aligned indicators:
1. Governance Oversight
2. Strategy & Resilience
3. Risk Management Integration
4. Metrics & Targets Quality
5. Transition Plan Credibility

Each indicator uses a `0-4` scale and a weighted aggregate score (`0-100`).


In [ ]:
COMMITMENT_SYSTEM_PROMPT = """
You are an ESG review expert focused on climate disclosures.
Evaluate climate-commitment quality using only provided excerpts.
Follow the rubric strictly and avoid unsupported assumptions.
Return one valid JSON object only.
""".strip()

TCFD_COMMITMENT_RUBRIC: Dict[str, Dict[str, Any]] = {
    "governance_oversight": {
        "tcfd_ref": "TCFD Governance (a,b)",
        "weight": 0.20,
        "what_to_check": "Board/senior-management oversight, role clarity, accountability cadence, incentive linkage.",
        "score_levels": {
            0: "No meaningful governance disclosure.",
            1: "Generic governance statement; no role detail or review cadence.",
            2: "Some role attribution and periodic review, but weak accountability details.",
            3: "Clear board/management responsibilities, governance process, and regular review evidence.",
            4: "Strong governance architecture with role clarity, decision rights, oversight cadence, and incentive/accountability linkage.",
        },
        "retrieval_terms": ["board", "committee", "oversight", "management", "governance", "accountability", "executive compensation"],
    },
    "strategy_resilience": {
        "tcfd_ref": "TCFD Strategy (a,b,c)",
        "weight": 0.20,
        "what_to_check": "Climate risks/opportunities in strategy, scenario analysis, resilience narrative, business model implications.",
        "score_levels": {
            0: "No climate strategy or resilience discussion.",
            1: "High-level ambition language without strategic integration.",
            2: "Some strategy linkage but limited scenario/resilience specificity.",
            3: "Clear strategy linkage with scenario or resilience evidence and business implications.",
            4: "Robust, decision-useful resilience framing with scenario assumptions and strategic responses.",
        },
        "retrieval_terms": ["strategy", "scenario", "resilience", "transition", "physical risk", "opportunity", "business model"],
    },
    "risk_management_integration": {
        "tcfd_ref": "TCFD Risk Management (a,b,c)",
        "weight": 0.20,
        "what_to_check": "How climate risks are identified, assessed, prioritized, and integrated into enterprise risk management.",
        "score_levels": {
            0: "No risk-management process for climate topics.",
            1: "Mentions risk but process is vague.",
            2: "Partial process description with limited integration evidence.",
            3: "Defined risk process with integration into broader risk management.",
            4: "Comprehensive and operationalized process with prioritization logic and governance linkage.",
        },
        "retrieval_terms": ["risk management", "identify", "assess", "prioritize", "materiality", "ERM", "controls"],
    },
    "metrics_targets_quality": {
        "tcfd_ref": "TCFD Metrics & Targets (a,b,c)",
        "weight": 0.20,
        "what_to_check": "Coverage of Scope emissions, target boundary, baseline year, target year, methodology, and progress tracking.",
        "score_levels": {
            0: "No useful metrics/targets disclosure.",
            1: "Target claims without baseline, boundary, or tracking detail.",
            2: "Some metrics/targets present but incomplete boundary/method/progress information.",
            3: "Good metrics-target package with baseline, timeline, and progress discussion.",
            4: "High-quality metrics-target package with clear boundary, method transparency, and consistent progress evidence.",
        },
        "retrieval_terms": ["scope 1", "scope 2", "scope 3", "target", "baseline", "progress", "methodology", "ghg protocol"],
    },
    "transition_plan_credibility": {
        "tcfd_ref": "TCFD Strategy + Metrics & Targets (implementation quality)",
        "weight": 0.20,
        "what_to_check": "Execution credibility: milestones, capex/opex signals, supply-chain actions, delivery evidence, and constraints.",
        "score_levels": {
            0: "No practical transition plan evidence.",
            1: "Aspirational narrative with minimal implementation detail.",
            2: "Some actions listed but weak milestones/resources/accountability.",
            3: "Credible plan with concrete actions and progress indicators.",
            4: "Highly credible plan with milestones, resource commitment, delivery evidence, and transparent constraints.",
        },
        "retrieval_terms": ["transition plan", "roadmap", "milestone", "investment", "capex", "implementation", "supplier engagement"],
    },
}

COMMITMENT_OUTPUT_SCHEMA = {
    "company": "string",
    "report_year_focus": "string or null",
    "indicator_scores": {
        "governance_oversight": {
            "score_0_to_4": "number",
            "rating": "insufficient|basic|developing|strong|leading",
            "rationale": "string",
            "evidence_quotes": ["short direct snippets"],
        },
        "strategy_resilience": {
            "score_0_to_4": "number",
            "rating": "insufficient|basic|developing|strong|leading",
            "rationale": "string",
            "evidence_quotes": ["short direct snippets"],
        },
        "risk_management_integration": {
            "score_0_to_4": "number",
            "rating": "insufficient|basic|developing|strong|leading",
            "rationale": "string",
            "evidence_quotes": ["short direct snippets"],
        },
        "metrics_targets_quality": {
            "score_0_to_4": "number",
            "rating": "insufficient|basic|developing|strong|leading",
            "rationale": "string",
            "evidence_quotes": ["short direct snippets"],
        },
        "transition_plan_credibility": {
            "score_0_to_4": "number",
            "rating": "insufficient|basic|developing|strong|leading",
            "rationale": "string",
            "evidence_quotes": ["short direct snippets"],
        },
    },
    "overall_score_100": "number",
    "overall_grade": "AAA|AA|A|BBB|BB|B|CCC",
    "key_strengths": ["string"],
    "key_gaps": ["string"],
    "semantic_analysis": {
        "commitment_tone": "aspirational|balanced|execution-oriented",
        "specificity_level": "low|medium|high",
        "delivery_signal": "low|medium|high",
        "greenwashing_risk": "low|medium|high",
        "narrative_consistency": "low|medium|high",
        "distinctive_language": ["short phrase"],
    },
    "confidence": "low|medium|high",
    "notes": "string",
}


def _indicator_keys() -> List[str]:
    return list(TCFD_COMMITMENT_RUBRIC.keys())


def _grade_from_score(score_100: float) -> str:
    if score_100 >= 90:
        return "AAA"
    if score_100 >= 80:
        return "AA"
    if score_100 >= 70:
        return "A"
    if score_100 >= 60:
        return "BBB"
    if score_100 >= 50:
        return "BB"
    if score_100 >= 40:
        return "B"
    return "CCC"


def _safe_float(x: Any) -> float | None:
    try:
        v = float(x)
    except Exception:
        return None
    if v < 0:
        v = 0.0
    if v > 4:
        v = 4.0
    return v


def _build_rubric_block() -> str:
    blocks = []
    for k, spec in TCFD_COMMITMENT_RUBRIC.items():
        levels = spec.get("score_levels", {})
        blocks.append(
            f"""
Indicator: {k}
TCFD reference: {spec['tcfd_ref']}
Weight: {spec['weight']}
What to check: {spec['what_to_check']}
Scoring guide:
0 = {levels.get(0,'')}
1 = {levels.get(1,'')}
2 = {levels.get(2,'')}
3 = {levels.get(3,'')}
4 = {levels.get(4,'')}
""".strip()
        )
    return "\\n\\n".join(blocks)


def _commitment_signal_score(indicator_key: str, text: str) -> float:
    spec = TCFD_COMMITMENT_RUBRIC[indicator_key]
    lower = text.lower()
    score = 0.0
    # indicator keywords
    for term in spec.get("retrieval_terms", []):
        if term.lower() in lower:
            score += 0.2
    # execution language
    if re.search(r"implemented|launched|reduced|invested|achieved|completed|progress", lower):
        score += 0.6
    # plan / target language
    if re.search(r"target|goal|net zero|by 2030|by 2040|by 2050|baseline", lower):
        score += 0.5
    # governance/risk language
    if re.search(r"board|committee|oversight|risk management|scenario", lower):
        score += 0.4
    return score


def _semantic_feature_snapshot(chunks: List[Dict[str, Any]]) -> Dict[str, Any]:
    text = " ".join([c.get("text", "") for c in chunks]).lower()
    aspirational = len(re.findall(r"aim|aspire|intend|plan to|seek to|strive", text))
    execution = len(re.findall(r"implemented|deployed|reduced|achieved|invested|completed", text))
    methodology = len(re.findall(r"ghg protocol|market-based|location-based|boundary|methodology", text))
    return {
        "aspirational_term_count": aspirational,
        "execution_term_count": execution,
        "methodology_term_count": methodology,
    }


def _format_commitment_chunks(chunks: List[Dict[str, Any]], max_chars: int = 680) -> str:
    if not chunks:
        return "<no chunks>"
    blocks = []
    for i, c in enumerate(chunks, 1):
        rr = c.get("rerank_score")
        rr_txt = f"{rr:.4f}" if isinstance(rr, (int, float)) else "n/a"
        blocks.append(
            f"[Chunk {i}] file={c['file_name']} chunk={c['chunk_id']} sim={c['similarity']:.4f} rerank={rr_txt}\\n"
            f"{c['text'][:max_chars]}"
        )
    return "\\n\\n".join(blocks)


def _build_commitment_hyde_prompt(company_name: str, indicator_key: str) -> str:
    spec = TCFD_COMMITMENT_RUBRIC[indicator_key]
    return f"""
Company: {company_name}
Indicator: {indicator_key}
TCFD reference: {spec['tcfd_ref']}
What to check: {spec['what_to_check']}
Likely terms: {', '.join(spec.get('retrieval_terms', []))}

Write 3 short hypothetical ESG-report style lines that would likely contain evidence for this indicator.
Use policy + implementation language.
Do not output JSON.
""".strip()


def _build_commitment_query(company_name: str, indicator_key: str, hyde_text: str = "") -> str:
    spec = TCFD_COMMITMENT_RUBRIC[indicator_key]
    base = f"""
Retrieve evidence for {company_name} on indicator {indicator_key}.
TCFD reference: {spec['tcfd_ref']}
Focus: {spec['what_to_check']}
Keywords: {' '.join(spec.get('retrieval_terms', []))}
""".strip()
    if hyde_text:
        base += "\n\nHypothetical evidence phrasing:\n" + hyde_text
    return base


def _build_commitment_table_query(company_name: str, indicator_key: str) -> str:
    spec = TCFD_COMMITMENT_RUBRIC[indicator_key]
    return f"""
Company {company_name} report sections for {indicator_key}.
Find sections and tables likely titled: governance, risk management, targets, progress, transition, metrics.
Look for year references, milestone language, baseline year, and implementation progress.
Keywords: {' '.join(spec.get('retrieval_terms', []))}
""".strip()


def _normalize_commitment_output(data: Dict[str, Any], company_name: str) -> Dict[str, Any]:
    if not isinstance(data, dict):
        data = {}
    data.setdefault("company", company_name)

    indicator_scores = data.get("indicator_scores")
    if not isinstance(indicator_scores, dict):
        indicator_scores = {}

    normalized_scores: Dict[str, Dict[str, Any]] = {}
    weighted = 0.0

    for k, spec in TCFD_COMMITMENT_RUBRIC.items():
        node = indicator_scores.get(k, {})
        if not isinstance(node, dict):
            node = {}

        s = _safe_float(node.get("score_0_to_4"))
        node["score_0_to_4"] = s
        if not isinstance(node.get("evidence_quotes"), list):
            node["evidence_quotes"] = []

        normalized_scores[k] = node

        if s is not None:
            weighted += (s / 4.0) * 100.0 * float(spec.get("weight", 0.0))

    data["indicator_scores"] = normalized_scores

    if not isinstance(data.get("overall_score_100"), (int, float)):
        data["overall_score_100"] = round(weighted, 1)

    if not data.get("overall_grade"):
        data["overall_grade"] = _grade_from_score(float(data["overall_score_100"]))

    if not isinstance(data.get("key_strengths"), list):
        data["key_strengths"] = []
    if not isinstance(data.get("key_gaps"), list):
        data["key_gaps"] = []
    if not isinstance(data.get("semantic_analysis"), dict):
        data["semantic_analysis"] = {}

    data.setdefault("confidence", "medium")
    data.setdefault("notes", "")
    return data


def _build_commitment_prompt(company_name: str, retrieval_package: Dict[str, Any]) -> str:
    indicator_blocks = []
    for k, spec in TCFD_COMMITMENT_RUBRIC.items():
        section = retrieval_package["by_indicator"].get(k, {})
        indicator_blocks.append(
            f"""
Indicator: {k}
TCFD reference: {spec['tcfd_ref']}
What to check: {spec['what_to_check']}
Retrieved evidence:
{_format_commitment_chunks(section.get('hits', []))}
""".strip()
        )

    return f"""
Task: Evaluate climate-commitment quality for {company_name} using TCFD-aligned scoring.

Output JSON schema:
{json.dumps(COMMITMENT_OUTPUT_SCHEMA, indent=2)}

Rubric:
{_build_rubric_block()}

Evidence by indicator:
{chr(10).join([''] + [b + chr(10) for b in indicator_blocks])}

Scoring rules:
1. Use only provided excerpts.
2. Score each indicator from 0 to 4 using rubric anchors.
3. Provide short evidence quotes per indicator.
4. If evidence is weak, score conservatively.
5. Provide semantic analysis (tone, specificity, delivery signal, greenwashing risk, consistency).
6. Return JSON only.
""".strip()


In [ ]:
COMMITMENT_HYDE_CACHE: Dict[str, str] = {}


def generate_commitment_hyde(company_name: str, indicator_key: str, model_name: str | None = None) -> str:
    model_name = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    cache_key = f"{company_name}::{indicator_key}::{model_name}"
    if cache_key in COMMITMENT_HYDE_CACHE:
        return COMMITMENT_HYDE_CACHE[cache_key]

    prompt = _build_commitment_hyde_prompt(company_name, indicator_key)
    txt = call_llm(system_prompt=HYDE_SYSTEM_PROMPT, user_prompt=prompt, model_name=model_name)
    txt = re.sub(r"\s+", " ", txt).strip()
    COMMITMENT_HYDE_CACHE[cache_key] = txt
    return txt


def retrieve_commitment_context(
    doc: Dict[str, Any],
    top_k_per_indicator: int = 4,
    retrieval_pool_size_per_indicator: int = 12,
    use_hyde_retrieval: bool = True,
    hyde_model_name: str | None = None,
    use_table_query_retrieval: bool = True,
    commitment_signal_weight: float = 0.08,
    table_signal_weight: float = 0.04,
) -> Dict[str, Any]:
    by_indicator: Dict[str, Dict[str, Any]] = {}
    merged_map: Dict[str, Dict[str, Any]] = {}
    hyde_errors: Dict[str, str] = {}

    for indicator_key in _indicator_keys():
        hyde_text = ""
        if use_hyde_retrieval:
            try:
                hyde_text = generate_commitment_hyde(doc["company"], indicator_key, model_name=hyde_model_name)
            except Exception as e:
                hyde_errors[indicator_key] = str(e)

        semantic_query = _build_commitment_query(doc["company"], indicator_key, hyde_text=hyde_text)
        table_query = _build_commitment_table_query(doc["company"], indicator_key)
        query_texts = [semantic_query]
        if use_table_query_retrieval:
            query_texts.append(table_query)

        pooled_hits = retrieve_chunks_multiquery(
            query_texts=query_texts,
            top_k_per_query=retrieval_pool_size_per_indicator,
            filter_doc_id=doc["doc_id"],
        )

        ranked_hits: List[Dict[str, Any]] = []
        for h in pooled_hits:
            item = dict(h)
            text = item.get("text", "")
            c_signal = _commitment_signal_score(indicator_key, text)
            t_signal = compute_table_signal_score(text)
            rr = float(item.get("similarity", 0.0))
            rr += commitment_signal_weight * c_signal
            rr += table_signal_weight * t_signal
            item["commitment_signal_score"] = round(c_signal, 3)
            item["table_signal_score"] = round(t_signal, 3)
            item["rerank_score"] = round(rr, 6)
            ranked_hits.append(item)

        ranked_hits.sort(key=lambda x: x["rerank_score"], reverse=True)
        final_hits = ranked_hits[:top_k_per_indicator]

        by_indicator[indicator_key] = {
            "query_texts": query_texts,
            "hyde_text": hyde_text,
            "hits": final_hits,
        }

        for h in final_hits:
            key = f"{h['file_name']}::{h['chunk_id']}"
            prev = merged_map.get(key)
            if prev is None or h["rerank_score"] > prev.get("rerank_score", 0.0):
                merged_map[key] = h

    merged_hits = sorted(merged_map.values(), key=lambda x: x.get("rerank_score", x.get("similarity", 0.0)), reverse=True)
    return {
        "by_indicator": by_indicator,
        "merged_hits": merged_hits,
        "hyde_errors": hyde_errors,
        "semantic_feature_snapshot": _semantic_feature_snapshot(merged_hits[:16]),
    }


def commitment_score_for_doc(
    doc: Dict[str, Any],
    model_name: str | None = None,
    top_k_per_indicator: int = 4,
    retrieval_pool_size_per_indicator: int = 12,
    use_hyde_retrieval: bool = True,
    hyde_model_name: str | None = None,
    use_table_query_retrieval: bool = True,
    commitment_signal_weight: float = 0.08,
    table_signal_weight: float = 0.04,
) -> Dict[str, Any]:
    retrieval_package = retrieve_commitment_context(
        doc,
        top_k_per_indicator=top_k_per_indicator,
        retrieval_pool_size_per_indicator=retrieval_pool_size_per_indicator,
        use_hyde_retrieval=use_hyde_retrieval,
        hyde_model_name=hyde_model_name,
        use_table_query_retrieval=use_table_query_retrieval,
        commitment_signal_weight=commitment_signal_weight,
        table_signal_weight=table_signal_weight,
    )

    if not retrieval_package["merged_hits"]:
        return {
            "company": doc["company"],
            "error": "No retrieved commitment evidence",
            "_rag_meta": {
                "provider": "colab_ai",
                "doc_id": doc["doc_id"],
                "file_name": doc["file_name"],
                "source_type": doc["source_type"],
            },
        }

    user_prompt = _build_commitment_prompt(doc["company"], retrieval_package)
    t0 = time.time()
    raw = call_llm(system_prompt=COMMITMENT_SYSTEM_PROMPT, user_prompt=user_prompt, model_name=model_name)
    elapsed = time.time() - t0

    try:
        parsed = parse_model_json(raw)
    except Exception as e:
        parsed = {
            "company": doc["company"],
            "parse_error": str(e),
            "raw_model_output": raw[:8000],
        }

    data = _normalize_commitment_output(parsed, company_name=doc["company"])

    requested_model = model_name or os.getenv("COLAB_AI_MODEL_NAME", "google/gemini-2.5-pro")
    actual_model = _COLAB_AI_CACHE.get("last_model_used") or requested_model

    indicator_hit_counts = {
        k: len(v.get("hits", [])) for k, v in retrieval_package["by_indicator"].items()
    }

    data["_rag_meta"] = {
        "provider": "colab_ai",
        "model": actual_model,
        "requested_model": requested_model,
        "model_attempts": _COLAB_AI_CACHE.get("last_model_attempts", []),
        "doc_id": doc["doc_id"],
        "file_name": doc["file_name"],
        "source_type": doc["source_type"],
        "top_k_per_indicator": top_k_per_indicator,
        "retrieval_pool_size_per_indicator": retrieval_pool_size_per_indicator,
        "use_hyde_retrieval": use_hyde_retrieval,
        "hyde_model_name": hyde_model_name or requested_model,
        "use_table_query_retrieval": use_table_query_retrieval,
        "commitment_signal_weight": commitment_signal_weight,
        "table_signal_weight": table_signal_weight,
        "indicator_hit_counts": indicator_hit_counts,
        "semantic_feature_snapshot": retrieval_package.get("semantic_feature_snapshot", {}),
        "latency_sec": round(elapsed, 2),
    }
    return data


## Step 14. Run Commitment Scoring (One Company + Batch)

We first run one company, then score the four companies with the same rubric.


In [ ]:
COMMITMENT_TOP_K_PER_INDICATOR = 4
COMMITMENT_RETRIEVAL_POOL_SIZE_PER_INDICATOR = 12
COMMITMENT_USE_HYDE_RETRIEVAL = True
COMMITMENT_HYDE_MODEL_NAME = "google/gemini-2.5-flash-lite"
COMMITMENT_USE_TABLE_QUERY_RETRIEVAL = True
COMMITMENT_SIGNAL_WEIGHT = 0.08
COMMITMENT_TABLE_SIGNAL_WEIGHT = 0.04

RUN_COMMITMENT_ONE = bool(IN_COLAB)
RUN_COMMITMENT_BATCH = bool(IN_COLAB)

commitment_single_result = None
if RUN_COMMITMENT_ONE and documents:
    preferred_order = ["Amazon", "Google", "Apple", "Meta"]
    selected = None
    for name in preferred_order:
        selected = next((d for d in documents if d["company"] == name), None)
        if selected:
            break

    print("Running commitment scoring for:", selected["company"], "|", selected["file_name"])
    commitment_single_result = commitment_score_for_doc(
        selected,
        model_name=COLAB_AI_MODEL_NAME,
        top_k_per_indicator=COMMITMENT_TOP_K_PER_INDICATOR,
        retrieval_pool_size_per_indicator=COMMITMENT_RETRIEVAL_POOL_SIZE_PER_INDICATOR,
        use_hyde_retrieval=COMMITMENT_USE_HYDE_RETRIEVAL,
        hyde_model_name=COMMITMENT_HYDE_MODEL_NAME,
        use_table_query_retrieval=COMMITMENT_USE_TABLE_QUERY_RETRIEVAL,
        commitment_signal_weight=COMMITMENT_SIGNAL_WEIGHT,
        table_signal_weight=COMMITMENT_TABLE_SIGNAL_WEIGHT,
    )
    print(json.dumps(commitment_single_result, indent=2, ensure_ascii=False)[:5000])
else:
    print("Skipping commitment one-company run (not in Colab or no documents).")


commitment_results: List[Dict[str, Any]] = []
if RUN_COMMITMENT_BATCH:
    for doc in documents:
        if doc["company"] not in {"Amazon", "Google", "Apple", "Meta"}:
            continue
        print(f"\n=== Commitment scoring: {doc['company']} | {doc['file_name']} ===")
        try:
            r = commitment_score_for_doc(
                doc,
                model_name=COLAB_AI_MODEL_NAME,
                top_k_per_indicator=COMMITMENT_TOP_K_PER_INDICATOR,
                retrieval_pool_size_per_indicator=COMMITMENT_RETRIEVAL_POOL_SIZE_PER_INDICATOR,
                use_hyde_retrieval=COMMITMENT_USE_HYDE_RETRIEVAL,
                hyde_model_name=COMMITMENT_HYDE_MODEL_NAME,
                use_table_query_retrieval=COMMITMENT_USE_TABLE_QUERY_RETRIEVAL,
                commitment_signal_weight=COMMITMENT_SIGNAL_WEIGHT,
                table_signal_weight=COMMITMENT_TABLE_SIGNAL_WEIGHT,
            )
        except Exception as e:
            r = {
                "company": doc["company"],
                "error": str(e),
                "_rag_meta": {
                    "provider": "colab_ai",
                    "doc_id": doc["doc_id"],
                    "file_name": doc["file_name"],
                    "source_type": doc["source_type"],
                },
            }
        commitment_results.append(r)

    print("\nCommitment batch complete:", len(commitment_results), "result(s)")
else:
    print("RUN_COMMITMENT_BATCH = False (set Colab runtime to run this stage).")


## Step 15. Build Commitment Score Table


In [ ]:
def _get_indicator_score(row: Dict[str, Any], key: str) -> float | None:
    node = (row.get("indicator_scores") or {}).get(key, {})
    try:
        if node.get("score_0_to_4") is None:
            return None
        return float(node.get("score_0_to_4"))
    except Exception:
        return None


commitment_summary_df = pd.DataFrame(
    [
        {
            "company": r.get("company"),
            "overall_score_100": r.get("overall_score_100"),
            "overall_grade": r.get("overall_grade"),
            "governance_oversight_score_0_4": _get_indicator_score(r, "governance_oversight"),
            "strategy_resilience_score_0_4": _get_indicator_score(r, "strategy_resilience"),
            "risk_management_integration_score_0_4": _get_indicator_score(r, "risk_management_integration"),
            "metrics_targets_quality_score_0_4": _get_indicator_score(r, "metrics_targets_quality"),
            "transition_plan_credibility_score_0_4": _get_indicator_score(r, "transition_plan_credibility"),
            "tone": ((r.get("semantic_analysis") or {}).get("commitment_tone")),
            "specificity_level": ((r.get("semantic_analysis") or {}).get("specificity_level")),
            "delivery_signal": ((r.get("semantic_analysis") or {}).get("delivery_signal")),
            "greenwashing_risk": ((r.get("semantic_analysis") or {}).get("greenwashing_risk")),
            "confidence": r.get("confidence"),
            "error": r.get("error"),
            "parse_error": r.get("parse_error"),
        }
        for r in commitment_results
    ]
)

commitment_summary_df


## Step 16. Save Commitment Scoring Outputs


In [ ]:
commitment_timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

commitment_json_path = OUTPUT_DIR / "climate_commitment_scores.json"
commitment_csv_path = OUTPUT_DIR / "climate_commitment_scores.csv"
commitment_run_record_path = OUTPUT_DIR / f"climate_commitment_run_record_{commitment_timestamp}.json"

with open(commitment_json_path, "w", encoding="utf-8") as f:
    json.dump(commitment_results, f, indent=2, ensure_ascii=False)

commitment_summary_df.to_csv(commitment_csv_path, index=False)

commitment_run_record = {
    "run_timestamp_utc": commitment_timestamp,
    "provider": "colab_ai",
    "model": COLAB_AI_MODEL_NAME,
    "model_fallbacks": COLAB_AI_FALLBACK_MODELS,
    "rubric_indicators": _indicator_keys(),
    "tcfd_alignment": {k: v["tcfd_ref"] for k, v in TCFD_COMMITMENT_RUBRIC.items()},
    "top_k_per_indicator": COMMITMENT_TOP_K_PER_INDICATOR,
    "retrieval_pool_size_per_indicator": COMMITMENT_RETRIEVAL_POOL_SIZE_PER_INDICATOR,
    "use_hyde_retrieval": COMMITMENT_USE_HYDE_RETRIEVAL,
    "hyde_model_name": COMMITMENT_HYDE_MODEL_NAME,
    "use_table_query_retrieval": COMMITMENT_USE_TABLE_QUERY_RETRIEVAL,
    "commitment_signal_weight": COMMITMENT_SIGNAL_WEIGHT,
    "table_signal_weight": COMMITMENT_TABLE_SIGNAL_WEIGHT,
    "target_companies": ["Amazon", "Google", "Apple", "Meta"],
    "doc_count_loaded": len(documents),
    "doc_files": [d["file_name"] for d in documents],
    "vector_collection": COLLECTION_NAME,
    "embedding_model": EMBED_MODEL_NAME,
    "commitment_json_path": str(commitment_json_path),
    "commitment_csv_path": str(commitment_csv_path),
}

with open(commitment_run_record_path, "w", encoding="utf-8") as f:
    json.dump(commitment_run_record, f, indent=2, ensure_ascii=False)

print("Saved:", commitment_json_path)
print("Saved:", commitment_csv_path)
print("Saved:", commitment_run_record_path)
